In [4]:
import pandas as pd 

df = pd.read_csv(r"https://cdn.enqurious.com/documents/aa1ba063-9ee6-4d79-bfd3-1b1503609f56_Globalmart_reviews.csv")

df.head()

,review_id,review_text,order_id,customer_id,reviews_rating
0,1,I initially had trouble deciding between the p...,CA-2016-152156,CG-12520,5.0
1,2,Allow me to preface this with a little history...,CA-2016-138688,DV-13045,5.0
2,3,I am enjoying it so far. Great for reading. Ha...,US-2015-108966,SO-20335,4.0
3,4,I bought one of the first Paperwhites and have...,CA-2014-115812,BH-11710,5.0
4,5,I have to say upfront - I don't like coroporat...,CA-2017-114412,AA-10480,5.0


## Task 1: Clean Customer Review Text

In [5]:
import re

# Define the function to clean text
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    return text.strip().lower()  # Convert to lowercase and strip leading/trailing spaces

# Apply the cleaning function to the 'review_text' column
df['cleaned_reviews'] = df['review_text'].apply(clean_text)

# Display a preview of the updated DataFrame with the new 'cleaned_reviews' column
df[['review_text', 'cleaned_reviews']].head()


,review_text,cleaned_reviews
0,I initially had trouble deciding between the p...,i initially had trouble deciding between the p...
1,Allow me to preface this with a little history...,allow me to preface this with a little history...
2,I am enjoying it so far. Great for reading. Ha...,i am enjoying it so far great for reading had ...
3,I bought one of the first Paperwhites and have...,i bought one of the first paperwhites and have...
4,I have to say upfront - I don't like coroporat...,i have to say upfront i dont like coroporate h...


## Task 2: Extract Mentions of Product Issues

In [6]:
# Define the regex pattern to match product issues
issue_pattern = r'\b(broken|defective|damaged)\b'

# Apply the pattern to the 'cleaned_reviews' column and create a new column 'mentions_issue'
df['mentions_issue'] = df['cleaned_reviews'].str.contains(issue_pattern, flags=re.IGNORECASE)

# Display a preview of the DataFrame with the new 'mentions_issue' column
df[['cleaned_reviews', 'mentions_issue']].head()


C:\Users\virin\AppData\Local\Temp\ipykernel_2560\2782132560.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['mentions_issue'] = df['cleaned_reviews'].str.contains(issue_pattern, flags=re.IGNORECASE)


,cleaned_reviews,mentions_issue
0,i initially had trouble deciding between the p...,False
1,allow me to preface this with a little history...,False
2,i am enjoying it so far great for reading had ...,False
3,i bought one of the first paperwhites and have...,False
4,i have to say upfront i dont like coroporate h...,False


In [7]:
df.mentions_issue.value_counts()

mentions_issue
False    1591
True        6
Name: count, dtype: int64

## Task 3: Extract Mentions of Contact Numbers

In [8]:
# Define the regex pattern for phone numbers
phone_pattern = r'\b(?:\+?\d{1,3})?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b'

# Extract phone numbers and create a new column 'contact_numbers'
df['contact_numbers'] = df['cleaned_reviews'].str.extract(f'({phone_pattern})', expand=False)

df[['cleaned_reviews', 'contact_numbers']].head()


,cleaned_reviews,contact_numbers
0,i initially had trouble deciding between the p...,NaN
1,allow me to preface this with a little history...,NaN
2,i am enjoying it so far great for reading had ...,NaN
3,i bought one of the first paperwhites and have...,NaN
4,i have to say upfront i dont like coroporate h...,NaN


In [9]:
# Check if any contact numbers were found and display a preview
contact_numbers_count = df['contact_numbers'].notna().sum()

In [10]:
contact_numbers_count

2

## Task 4: Count Keywords in Reviews

In [11]:
# Function to count keyword occurrences in text
def count_keyword(text, keyword):
    return len(re.findall(rf'\b{keyword}\b', text, flags=re.IGNORECASE))

# Add columns for keyword counts
keywords = ['excellent', 'poor', 'refund']
for keyword in keywords:
    df[f'count_{keyword}'] = df['cleaned_reviews'].apply(lambda x: count_keyword(x, keyword))

# Display a preview of the updated DataFrame with keyword counts
df[['cleaned_reviews', 'count_excellent', 'count_poor', 'count_refund']].head()


,cleaned_reviews,count_excellent,count_poor,count_refund
0,i initially had trouble deciding between the p...,0,0,0
1,allow me to preface this with a little history...,0,0,0
2,i am enjoying it so far great for reading had ...,0,0,0
3,i bought one of the first paperwhites and have...,0,0,0
4,i have to say upfront i dont like coroporate h...,0,0,0


In [12]:
df.count_excellent.value_counts()

count_excellent
0    1512
1      75
2       9
3       1
Name: count, dtype: int64

In [13]:
df.count_poor.value_counts()

count_poor
0    1590
1       5
2       2
Name: count, dtype: int64

In [14]:
df.count_refund.value_counts()

count_refund
0    1592
1       5
Name: count, dtype: int64

## Task 5: Validate Review Length

In [15]:
# Function to validate review length
def validate_review_length(text):
    word_count = len(re.findall(r'\b\w+\b', text))  # Count words in the text
    return 10 <= word_count <= 100

# Apply the validation function
df['valid_review'] = df['cleaned_reviews'].apply(validate_review_length)

# Display a preview of the DataFrame with the new 'valid_review' column
df[['cleaned_reviews', 'valid_review']].head()


,cleaned_reviews,valid_review
0,i initially had trouble deciding between the p...,False
1,allow me to preface this with a little history...,False
2,i am enjoying it so far great for reading had ...,True
3,i bought one of the first paperwhites and have...,False
4,i have to say upfront i dont like coroporate h...,False


In [16]:
df.valid_review.value_counts()

valid_review
False    813
True     784
Name: count, dtype: int64